# Modal 1x H100 Full Repro (trainzeh4-workbanget)

Notebook ini menjalankan semuanya lewat notebook:
1. Download dataset dari Kaggle ke workspace Modal
2. Clone repo ke workspace Modal
3. Setup .venv dan install dependency profile
4. Download parent teacher model dari Hugging Face
5. Gabung metadata.csv + metadata_indsp.csv
6. Prepare dataset Arrow
7. Setup W&B login via env secret
8. Distillation + full training di 1x H100 (optimized)
9. Push final checkpoint ke Hugging Face Hub (repo baru, private)

Semua command Python dijalankan via: uv run --python .venv/bin/python

Secret env dibaca dari os.environ dan harus di-inject oleh Modal secret.
Contoh pattern: @app.function(secrets=[modal.Secret.from_name("Kegel")])

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

def require_env(var_name: str, modal_secret_name: str) -> str:
    value = os.environ.get(var_name)
    if value:
        return value
    raise EnvironmentError(
        f"{var_name} belum ter-set. Inject via Modal secret '{modal_secret_name}', contoh:\n"
        f"@app.function(secrets=[modal.Secret.from_name('{modal_secret_name}')])"
    )

# ================= User Config =================
REPO_URL = "https://github.com/AneKazek/malesbgt.git"
REPO_BRANCH = "nazwaaa"

# Modal secret names (sesuaikan dengan dashboard Modal kamu)
KAGGLE_SECRET_NAME = "Kegel"
WANDB_SECRET_NAME = "Wandebe"
HF_SECRET_NAME = "Haeface"

KAGGLE_USERNAME = require_env("KAGGLE_USERNAME", KAGGLE_SECRET_NAME)
KAGGLE_KEY = require_env("KAGGLE_KEY", KAGGLE_SECRET_NAME)
WANDB_API_KEY = require_env("WANDB_API_KEY", WANDB_SECRET_NAME)
HF_TOKEN = require_env("HF_TOKEN", HF_SECRET_NAME)

WANDB_ENTITY = "haidarmuhammaddzaky-institut-teknologi-sepuluh-nopember"
WANDB_PROJECT = "kaceveozon"

HF_REPO_ID = "Eempostor/F5-TTS-INDO-FINETUNE-V2"
HF_CKPT_FILENAME = "f5_tts_indo_v2.pt"

# Optional output repo naming for pushed checkpoints (new private repo always)
HF_OUTPUT_REPO_PREFIX = os.environ.get("HF_OUTPUT_REPO_PREFIX", "kcv-tts-modal-h100-ckpt")
HF_OUTPUT_REPO_OWNER = os.environ.get("HF_OUTPUT_REPO_OWNER")
HF_OUTPUT_REPO_ID = os.environ.get("HF_OUTPUT_REPO_ID")

# Kaggle dataset source (Modal flow)
KAGGLE_DATASET = "benedictusryugunawan/tts-indo"
KAGGLE_DATASET_SUBDIR = "data"
USE_METADATA_INDSP = False

# Modal workspace paths
WORKDIR = Path(os.environ.get("MODAL_WORKDIR", "/root/modal-workdir"))
DATASET_ROOT = WORKDIR / "datasets" / "tts_indo"
DATASET_DATA_DIR = DATASET_ROOT / KAGGLE_DATASET_SUBDIR

CSV_1 = DATASET_DATA_DIR / "metadata.csv"
CSV_2 = (DATASET_DATA_DIR / "metadata_indsp.csv") if USE_METADATA_INDSP else None

REPO_DIR = WORKDIR / "kcv-tts"
VENV_DIR = REPO_DIR / ".venv"
VENV_PY = VENV_DIR / "bin/python"

HF_OUT_DIR = REPO_DIR / "ckpts/hf/Eempostor_F5-TTS-INDO-FINETUNE-V2"
TEACHER_CKPT = HF_OUT_DIR / HF_CKPT_FILENAME
MERGED_CSV = REPO_DIR / "data/metadata_merged.csv"
PREPARED_DATASET_DIR = REPO_DIR / "data/datasetku_pinyin"

# H100-oriented defaults (non-smoke)
H100_MIXED_PRECISION = "bf16"
H100_DISTILL_BATCH_FRAMES = 12800
H100_FULL_BATCH_FRAMES = 25600
H100_MAX_SAMPLES = 64
H100_NUM_WORKERS = int(os.environ.get("TRAIN_NUM_WORKERS", "8"))
H100_USE_FLASH_ATTN = os.environ.get("H100_USE_FLASH_ATTN", "1") == "1"

def run_cmd(cmd, cwd=None, env=None, timeout=None):
    printable = cmd if isinstance(cmd, str) else " ".join(str(x) for x in cmd)
    print("\n$", printable)
    if timeout is not None:
        print(f"(timeout={timeout}s)")
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
        text=True,
        timeout=timeout,
    )

def run_py(args, cwd=None, env=None, timeout=None):
    # --no-sync mencegah uv mengubah isi .venv saat eksekusi python command.
    return run_cmd(["uv", "run", "--no-sync", "--python", str(VENV_PY), "python", *args], cwd=cwd, env=env, timeout=timeout)

WORKDIR.mkdir(parents=True, exist_ok=True)

print("Config siap.")
print("WORKDIR:", WORKDIR)
print("Dataset root:", DATASET_ROOT)
print("CSV_1 target:", CSV_1)
print("CSV_2 enabled:", CSV_2 is not None)
print("Modal secret mapping:", {
    "kaggle": KAGGLE_SECRET_NAME,
    "wandb": WANDB_SECRET_NAME,
    "huggingface": HF_SECRET_NAME,
})
print("H100 preset:", {
    "mixed_precision": H100_MIXED_PRECISION,
    "distill_batch_frames": H100_DISTILL_BATCH_FRAMES,
    "full_batch_frames": H100_FULL_BATCH_FRAMES,
    "max_samples": H100_MAX_SAMPLES,
    "num_workers": H100_NUM_WORKERS,
    "use_flash_attn": H100_USE_FLASH_ATTN,
})

Config siap.
CSV_1 exists: True
CSV_2 enabled: False


In [ ]:
# 1) Download dataset dari Kaggle (Modal) + clone repo
DATASET_ROOT.mkdir(parents=True, exist_ok=True)

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json = kaggle_dir / "kaggle.json"
kaggle_json.write_text(
    json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}),
    encoding="utf-8",
)
run_cmd(["chmod", "600", str(kaggle_json)])

run_cmd(["python3", "-m", "pip", "install", "-U", "kaggle"])

kaggle_download_args = [
    "datasets",
    "download",
    "-d",
    KAGGLE_DATASET,
    "-p",
    str(DATASET_ROOT),
    "--unzip",
]

candidate_cmds = []
kaggle_exe = shutil.which("kaggle")
if kaggle_exe:
    candidate_cmds.append([kaggle_exe, *kaggle_download_args])

# Kaggle >= 2.0 kadang tidak expose `python -m kaggle`, jadi fallback ke kaggle.cli jika perlu.
candidate_cmds.append(["python3", "-m", "kaggle.cli", *kaggle_download_args])

last_err = None
for cmd in candidate_cmds:
    try:
        run_cmd(cmd)
        last_err = None
        break
    except subprocess.CalledProcessError as e:
        print(f"Kaggle command gagal, mencoba metode lain: {cmd}")
        last_err = e

if last_err is not None:
    raise RuntimeError("Gagal download dataset via semua metode invoke Kaggle CLI.") from last_err

if not CSV_1.exists():
    fallback_csv = next(iter(sorted(DATASET_ROOT.glob("**/metadata.csv"))), None)
    if fallback_csv is None:
        raise FileNotFoundError(f"metadata.csv tidak ditemukan setelah download: {DATASET_ROOT}")

    CSV_1 = fallback_csv
    if USE_METADATA_INDSP:
        indsp_cand = fallback_csv.parent / "metadata_indsp.csv"
        CSV_2 = indsp_cand if indsp_cand.exists() else None

print("CSV_1 resolved:", CSV_1)
print("CSV_2 resolved:", CSV_2)
run_cmd(["ls", "-lah", str(DATASET_ROOT)])

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

run_cmd(["git", "clone", "--recursive", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])
run_cmd(["ls", "-lah", str(REPO_DIR)])


$ git clone --recursive --branch main https://github.com/AneKazek/malesbgt.git /kaggle/temp/kcv-tts


Cloning into '/kaggle/temp/kcv-tts'...



$ ls -lah /kaggle/temp/kcv-tts
total 1.1M
drwxr-xr-x 6 root root 4.0K Mar 29 17:49 .
drwxr-xr-x 3 root root 4.0K Mar 29 17:49 ..
-rw-r--r-- 1 root root 1.3K Mar 29 17:49 DEPENDENCY_PROFILES.md
-rw-r--r-- 1 root root  865 Mar 29 17:49 Dockerfile
-rw-r--r-- 1 root root  13K Mar 29 17:49 F5TTS_DATASET_REQUIREMENTS.md
drwxr-xr-x 8 root root 4.0K Mar 29 17:49 .git
drwxr-xr-x 4 root root 4.0K Mar 29 17:49 .github
-rw-r--r-- 1 root root 3.2K Mar 29 17:49 .gitignore
-rw-r--r-- 1 root root  115 Mar 29 17:49 .gitmodules
-rw-r--r-- 1 root root 6.4K Mar 29 17:49 LEMAS_vs_F5TTS_ARCHITECTURE.md
-rw-r--r-- 1 root root 1.1K Mar 29 17:49 LICENSE
drwxr-xr-x 2 root root 4.0K Mar 29 17:49 notebooks
-rw-r--r-- 1 root root  413 Mar 29 17:49 .pre-commit-config.yaml
-rw-r--r-- 1 root root 1.7K Mar 29 17:49 pyproject.toml
-rw-r--r-- 1 root root 9.6K Mar 29 17:49 README.md
-rw-r--r-- 1 root root  719 Mar 29 17:49 requirements-kaggle-torch210.txt
-rw-r--r-- 1 root root  805 Mar 29 17:49 requirements-py311-torch

CompletedProcess(args=['ls', '-lah', '/kaggle/temp/kcv-tts'], returncode=0)

In [5]:
# 2) Setup .venv + install dependency profile (local/kaggle parity)
if shutil.which("uv") is None:
    run_cmd(["python3", "-m", "pip", "install", "-U", "uv"])

TARGET_PY_MM = "3.11"
TARGET_PY = Path(f"/usr/bin/python{TARGET_PY_MM}")

run_cmd(["apt-get", "update", "-y"])
run_cmd([
    "apt-get",
    "install",
    "-y",
    f"python{TARGET_PY_MM}",
    f"python{TARGET_PY_MM}-venv",
    f"python{TARGET_PY_MM}-dev",
    "build-essential",
])

if not TARGET_PY.exists():
    raise FileNotFoundError(f"Interpreter target tidak ditemukan: {TARGET_PY}")

run_cmd(["uv", "venv", "--python", str(TARGET_PY), "--clear", str(VENV_DIR)])
run_cmd([str(VENV_PY), "-c", "import sys; print('venv python =', sys.version); assert sys.version_info[:2] == (3, 11)"])

# Setuptools<82 menghindari masalah build extension tertentu (mamba/causal-conv1d).
run_cmd([
    "uv",
    "pip",
    "install",
    "--python",
    str(VENV_PY),
    "--upgrade",
    "pip",
    "wheel",
    "setuptools<82",
])

req_candidates = [
    REPO_DIR / "requirements-torch28-cu12-localmatch.txt",
    REPO_DIR / "requirements-kaggle-torch210.txt",
]

REQ_PROFILE = next((p for p in req_candidates if p.exists()), None)
if REQ_PROFILE is None:
    raise FileNotFoundError("Tidak menemukan file profile requirements untuk setup training.")

print("Using dependency profile:", REQ_PROFILE)
run_cmd([
    "uv",
    "pip",
    "install",
    "--python",
    str(VENV_PY),
    "--index-strategy",
    "unsafe-best-match",
    "-r",
    str(REQ_PROFILE),
])

# Final guard: pastikan stack torch sesuai target training mamba.
run_cmd([
    "uv",
    "pip",
    "install",
    "--python",
    str(VENV_PY),
    "--index-url",
    "https://download.pytorch.org/whl/cu128",
    "--extra-index-url",
    "https://pypi.org/simple",
    "--index-strategy",
    "unsafe-best-match",
    "--force-reinstall",
    "--no-cache-dir",
    "torch==2.8.0+cu128",
    "torchvision==0.23.0+cu128",
    "torchaudio==2.8.0+cu128",
    "nvidia-nccl-cu12==2.27.3",
    "nvidia-nvjitlink-cu12==12.8.93",
])

run_cmd([
    str(VENV_PY),
    "-c",
    "import torch, torchvision, torchaudio, setuptools; "
    "print('torch =', torch.__version__, 'cuda =', torch.version.cuda); "
    "print('torchvision =', torchvision.__version__); "
    "print('torchaudio =', torchaudio.__version__); "
    "print('setuptools =', setuptools.__version__)"
])


$ apt-get update -y
Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,473 kB]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu ja

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


Reading package lists...
Building dependency tree...
Reading state information...
build-essential is already the newest version (12.9ubuntu3).
The following additional packages will be installed:
  libpython3.11 libpython3.11-dev libpython3.11-minimal libpython3.11-stdlib
  python3.11-minimal
Suggested packages:
  binfmt-support
The following NEW packages will be installed:
  libpython3.11 libpython3.11-dev libpython3.11-minimal libpython3.11-stdlib
  python3.11 python3.11-dev python3.11-minimal python3.11-venv
0 upgraded, 8 newly installed, 0 to remove and 152 not upgraded.
Need to get 16.5 MB of archives.
After this operation, 58.4 MB of additional disk space will be used.
Get:1 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 libpython3.11-minimal amd64 3.11.15-1+jammy1 [887 kB]
Get:2 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 python3.11-minimal amd64 3.11.15-1+jammy1 [2,353 kB]
Get:3 https://ppa.launchpadcontent.net/deadsnakes/ppa

Using CPython 3.11.15 interpreter at: /usr/bin/python3.11
Creating virtual environment at: /kaggle/temp/kcv-tts/.venv
Activate with: source /kaggle/temp/kcv-tts/.venv/bin/activate



$ /kaggle/temp/kcv-tts/.venv/bin/python -c import sys; print('venv python =', sys.version); assert sys.version_info[:2] == (3, 11)
venv python = 3.11.15 (main, Mar  3 2026, 09:26:23) [GCC 11.4.0]

$ uv pip install --python /kaggle/temp/kcv-tts/.venv/bin/python --upgrade pip wheel setuptools<82


...

Resolved 138 packages in 48.15s
Prepared 136 packages in 1m 06s
Installed 136 packages in 430ms
 + absl-py==2.4.0
 + accelerate==1.13.0
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.4
 + aiosignal==1.4.0
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + antlr4-python3-runtime==4.9.3
 + anyio==4.13.0
 + attrs==26.1.0
 + audioread==3.1.0
 + bitsandbytes==0.49.2
 + boto3==1.42.78
 + botocore==1.42.78
 + cached-path==1.8.10
 + causal-conv1d==1.6.1+cu12torch2.8cxx11abitrue (from https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1+cu12torch2.8cxx11abiTRUE-cp311-cp311-linux_x86_64.whl)
 + certifi==2026.2.25
 + cffi==2.0.0
 + charset-normalizer==3.4.6
 + click==8.3.1
 + cryptography==46.0.6
 + datasets==4.8.4
 + decorator==5.2.1
 + dill==0.4.1
 + einops==0.8.2
 + einx==0.4.2
 + ema-pytorch==0.7.9
 + filelock==3.25.2
 + fire==0.7.1
 + frozendict==2.4.7
 + frozenlist==1.8.0
 + fsspec==2026.2.0
 + gitdb==4.0.12
 + gitpython==3.1.46
 + google-api-core==2.


$ uv pip install --python /kaggle/temp/kcv-tts/.venv/bin/python --index-url https://download.pytorch.org/whl/cu128 --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match --force-reinstall --no-cache-dir torch==2.8.0+cu128 torchvision==0.23.0+cu128 torchaudio==2.8.0+cu128 nvidia-nccl-cu12==2.27.3 nvidia-nvjitlink-cu12==12.8.93


Resolved 29 packages in 2.70s
Prepared 29 packages in 33.51s
Uninstalled 29 packages in 410ms
Installed 29 packages in 368ms
 ~ filelock==3.25.2
 - fsspec==2026.2.0
 + fsspec==2026.3.0
 ~ jinja2==3.1.6
 ~ markupsafe==3.0.3
 ~ mpmath==1.3.0
 ~ networkx==3.6.1
 ~ numpy==2.4.4
 ~ nvidia-cublas-cu12==12.8.4.1
 ~ nvidia-cuda-cupti-cu12==12.8.90
 ~ nvidia-cuda-nvrtc-cu12==12.8.93
 ~ nvidia-cuda-runtime-cu12==12.8.90
 ~ nvidia-cudnn-cu12==9.10.2.21
 ~ nvidia-cufft-cu12==11.3.3.83
 ~ nvidia-cufile-cu12==1.13.1.3
 ~ nvidia-curand-cu12==10.3.9.90
 ~ nvidia-cusolver-cu12==11.7.3.90
 ~ nvidia-cusparse-cu12==12.5.8.93
 ~ nvidia-cusparselt-cu12==0.7.1
 ~ nvidia-nccl-cu12==2.27.3
 ~ nvidia-nvjitlink-cu12==12.8.93
 ~ nvidia-nvtx-cu12==12.8.90
 ~ pillow==12.1.1
 - setuptools==81.0.0
 + setuptools==82.0.1
 ~ sympy==1.14.0
 ~ torch==2.8.0+cu128
 ~ torchaudio==2.8.0+cu128
 ~ torchvision==0.23.0+cu128
 ~ triton==3.4.0
 ~ typing-extensions==4.15.0



$ /kaggle/temp/kcv-tts/.venv/bin/python -c import torch, torchvision, torchaudio, setuptools; print('torch =', torch.__version__, 'cuda =', torch.version.cuda); print('torchvision =', torchvision.__version__); print('torchaudio =', torchaudio.__version__); print('setuptools =', setuptools.__version__)
torch = 2.8.0+cu128 cuda = 12.8
torchvision = 0.23.0+cu128
torchaudio = 2.8.0+cu128
setuptools = 82.0.1


CompletedProcess(args=['/kaggle/temp/kcv-tts/.venv/bin/python', '-c', "import torch, torchvision, torchaudio, setuptools; print('torch =', torch.__version__, 'cuda =', torch.version.cuda); print('torchvision =', torchvision.__version__); print('torchaudio =', torchaudio.__version__); print('setuptools =', setuptools.__version__)"], returncode=0)

In [ ]:
# 3) Validasi GPU di Modal (target: 1x H100)
run_py([
    "-c",
    "import torch; "
    "print('torch', torch.__version__); "
    "print('cuda_count', torch.cuda.device_count()); "
    "[print(i, torch.cuda.get_device_name(i), 'cc', torch.cuda.get_device_capability(i)) for i in range(torch.cuda.device_count())]; "
    "assert torch.cuda.device_count() >= 1, 'GPU tidak terdeteksi'",
])


$ uv run --no-sync --python /kaggle/temp/kcv-tts/.venv/bin/python python -c import torch; print('torch', torch.__version__); print('cuda_count', torch.cuda.device_count()); [print(i, torch.cuda.get_device_name(i)) for i in range(torch.cuda.device_count())]


torch 2.8.0+cu128
cuda_count 2
0 Tesla T4
1 Tesla T4


CompletedProcess(args=['uv', 'run', '--no-sync', '--python', '/kaggle/temp/kcv-tts/.venv/bin/python', 'python', '-c', "import torch; print('torch', torch.__version__); print('cuda_count', torch.cuda.device_count()); [print(i, torch.cuda.get_device_name(i)) for i in range(torch.cuda.device_count())]"], returncode=0)

In [7]:
# 4) Download parent teacher model dari HF
run_cmd(["uv", "pip", "install", "--python", str(VENV_PY), "-U", "huggingface_hub"])

download_script = "\n".join([
    "from pathlib import Path",
    "from huggingface_hub import hf_hub_download",
    f"repo_id = {HF_REPO_ID!r}",
    f"filename = {HF_CKPT_FILENAME!r}",
    f"out_dir = Path(r'{HF_OUT_DIR}')",
    "out_dir.mkdir(parents=True, exist_ok=True)",
    "path = hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(out_dir), local_dir_use_symlinks=False)",
    "print(path)",
])

run_py(["-c", download_script], cwd=REPO_DIR)
run_cmd(["ls", "-lah", str(HF_OUT_DIR)])


$ uv pip install --python /kaggle/temp/kcv-tts/.venv/bin/python -U huggingface_hub


Using Python 3.11.15 environment at: /kaggle/temp/kcv-tts/.venv
Resolved 22 packages in 264ms
Prepared 1 package in 56ms
Uninstalled 1 package in 1ms
Installed 1 package in 3ms
 - rich==13.9.4
 + rich==14.3.3



$ uv run --no-sync --python /kaggle/temp/kcv-tts/.venv/bin/python python -c from pathlib import Path
from huggingface_hub import hf_hub_download
repo_id = 'Eempostor/F5-TTS-INDO-FINETUNE-V2'
filename = 'f5_tts_indo_v2.pt'
out_dir = Path(r'/kaggle/temp/kcv-tts/ckpts/hf/Eempostor_F5-TTS-INDO-FINETUNE-V2')
out_dir.mkdir(parents=True, exist_ok=True)
path = hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(out_dir), local_dir_use_symlinks=False)
print(path)


/kaggle/temp/kcv-tts/.venv/lib/python3.11/site-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


/kaggle/temp/kcv-tts/ckpts/hf/Eempostor_F5-TTS-INDO-FINETUNE-V2/f5_tts_indo_v2.pt

$ ls -lah /kaggle/temp/kcv-tts/ckpts/hf/Eempostor_F5-TTS-INDO-FINETUNE-V2
total 1.3G
drwxr-xr-x 3 root root 4.0K Mar 29 17:52 .
drwxr-xr-x 3 root root 4.0K Mar 29 17:52 ..
drwxr-xr-x 3 root root 4.0K Mar 29 17:52 .cache
-rw-r--r-- 1 root root 1.3G Mar 29 17:52 f5_tts_indo_v2.pt


CompletedProcess(args=['ls', '-lah', '/kaggle/temp/kcv-tts/ckpts/hf/Eempostor_F5-TTS-INDO-FINETUNE-V2'], returncode=0)

In [9]:
# 5) Merge 2 CSV metadata -> audio_file|text (pipe-delimited)
import csv
import pandas as pd

def _normalize_metadata_df(df: pd.DataFrame, path: Path) -> pd.DataFrame:
    df.columns = [str(c).strip() for c in df.columns]

    audio_candidates = ["audio_file", "audio_path", "wav_path", "path", "file"]
    text_candidates = ["text", "transcript", "sentence", "normalized_text", "utterance"]

    audio_col = next((c for c in audio_candidates if c in df.columns), None)
    text_col = next((c for c in text_candidates if c in df.columns), None)

    if audio_col is None or text_col is None:
        raise ValueError(f"Kolom tidak cocok di {path}. Dapat: {list(df.columns)}")

    out = df[[audio_col, text_col]].copy()
    out.columns = ["audio_file", "text"]
    out["audio_file"] = out["audio_file"].astype(str).str.strip()
    out["text"] = out["text"].astype(str).str.strip()
    out = out[(out["audio_file"] != "") & (out["text"] != "")]

    source_dir = path.parent
    prefer_indsp = "indsp" in path.name.lower()

    def absolutize(p: str) -> str:
        raw = str(p).strip().strip('"').strip("'")
        pp = Path(raw).expanduser()
        if pp.is_absolute():
            return str(pp)

        has_dir = ("/" in raw) or ("\\" in raw)
        candidates = [source_dir / pp]

        # metadata_indsp sering berisi filename polos (tanpa folder), jadi coba prefix indsp/.
        if not has_dir:
            if prefer_indsp:
                candidates.insert(0, source_dir / "indsp" / pp)
            else:
                candidates.append(source_dir / "wavs" / pp)
                candidates.append(source_dir / "indsp" / pp)

        for cand in candidates:
            if cand.exists():
                return str(cand.resolve())

        return str(candidates[0].resolve())

    out["audio_file"] = out["audio_file"].map(absolutize)
    out["source_csv"] = str(path)
    return out

def _manual_parse_metadata(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8-sig", errors="replace") as f:
        first_non_empty = ""
        for ln in f:
            if ln.strip():
                first_non_empty = ln
                break
        f.seek(0)

        delim = "|" if first_non_empty.count("|") >= first_non_empty.count(",") else ","

        for i, ln in enumerate(f):
            ln = ln.strip()
            if not ln:
                continue
            if i == 0 and "audio_file" in ln.lower() and "text" in ln.lower():
                continue
            if delim not in ln:
                continue
            audio, text = ln.split(delim, 1)
            audio = audio.strip().strip('"')
            text = text.strip()
            if audio and text:
                rows.append((audio, text))

    df = pd.DataFrame(rows, columns=["audio_file", "text"])
    df["source_csv"] = str(path)
    return df

def load_metadata(path: Path | None) -> pd.DataFrame:
    if path is None:
        print("load_metadata: path None, skip.")
        return pd.DataFrame(columns=["audio_file", "text", "source_csv"])
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Metadata CSV tidak ditemukan: {path}")

    parse_attempts = [
        dict(
            sep="|",
            engine="python",
            dtype=str,
            keep_default_na=False,
            encoding="utf-8-sig",
            on_bad_lines="skip",
            quoting=csv.QUOTE_NONE,
        ),
        dict(
            sep=",",
            engine="python",
            dtype=str,
            keep_default_na=False,
            encoding="utf-8-sig",
            on_bad_lines="skip",
            quoting=csv.QUOTE_NONE,
        ),
    ]

    for kwargs in parse_attempts:
        try:
            df = pd.read_csv(path, **kwargs)
            out = _normalize_metadata_df(df, path)
            if len(out) > 0:
                return out
        except Exception:
            pass

    print(f"Parser fallback aktif untuk {path}")
    return _normalize_metadata_df(_manual_parse_metadata(path), path)

sources = [CSV_1]
if CSV_2 is not None:
    sources.append(CSV_2)
else:
    print("CSV_2/metadata_indsp dimatikan sementara. Hanya pakai CSV_1.")

dfs = [load_metadata(path) for path in sources]
merged = pd.concat(dfs, ignore_index=True).drop_duplicates(subset=["audio_file", "text"])

exists_mask = merged["audio_file"].map(lambda p: Path(p).exists())
missing = int((~exists_mask).sum())
if missing:
    print(f"Dropping {missing} rows with missing audio paths.")
merged = merged[exists_mask].copy()

MERGED_CSV.parent.mkdir(parents=True, exist_ok=True)
merged[["audio_file", "text"]].to_csv(MERGED_CSV, sep="|", index=False)

print("Merged rows (after drop missing):", len(merged))
print("Saved:", MERGED_CSV)
print("Source CSVs:", *sources)
print(merged[["audio_file", "text"]].head(5))

CSV_2/metadata_indsp dimatikan sementara. Hanya pakai CSV_1.
Merged rows (after drop missing): 4972
Saved: /kaggle/temp/kcv-tts/data/metadata_merged.csv
Source CSVs: /kaggle/input/datasets/benedictusryugunawan/tts-indo/data/metadata.csv
                                          audio_file  \
0  /kaggle/input/datasets/benedictusryugunawan/tt...   
1  /kaggle/input/datasets/benedictusryugunawan/tt...   
2  /kaggle/input/datasets/benedictusryugunawan/tt...   
3  /kaggle/input/datasets/benedictusryugunawan/tt...   
4  /kaggle/input/datasets/benedictusryugunawan/tt...   

                                                text  
0  Reaksi berbeda disampaikan penasihat hukum pen...  
1  Konflik eksekutif-legislatif sebenarnya adalah...  
2  Jenderal TNI Endriartono Sutarto memuji sosok ...  
3  Ketua fraksi PDIP Arifin Panigoro dan Yusril I...  
4  Yakni Gubernur Sutiyoso, Djaelani, Fauzi Bowo,...  


In [10]:
# 6) Siapkan data/Emilia_ZH_EN_pinyin/vocab.txt (minimal, hanya yang dibutuhkan prepare_csv_wavs.py)
EMILIA_VOCAB_DIR = REPO_DIR / "data" / "Emilia_ZH_EN_pinyin"
EMILIA_VOCAB_PATH = EMILIA_VOCAB_DIR / "vocab.txt"
EMILIA_VOCAB_DIR.mkdir(parents=True, exist_ok=True)

if EMILIA_VOCAB_PATH.exists() and EMILIA_VOCAB_PATH.stat().st_size > 0:
    print("Pretrained vocab sudah ada:", EMILIA_VOCAB_PATH)
else:
    run_cmd(["uv", "pip", "install", "--python", str(VENV_PY), "-U", "huggingface_hub"])

    vocab_fetch_script = "\n".join([
        "from pathlib import Path",
        "import shutil",
        "from huggingface_hub import hf_hub_download",
        f"target = Path(r'{EMILIA_VOCAB_PATH}')",
        "target.parent.mkdir(parents=True, exist_ok=True)",
        "candidates = [",
        "    ('SWivid/F5-TTS', 'F5TTS_Base/vocab.txt'),",
        "    ('SWivid/F5-TTS', 'F5TTS_v1_Base/vocab.txt'),",
        "]",
        "last_err = None",
        "for repo_id, filename in candidates:",
        "    try:",
        "        src = Path(hf_hub_download(repo_id=repo_id, filename=filename))",
        "        shutil.copy2(src, target)",
        "        print(f'Downloaded vocab from {repo_id}/{filename} -> {target}')",
        "        break",
        "    except Exception as e:",
        "        print(f'Gagal dari {repo_id}/{filename}: {e}')",
        "        last_err = e",
        "else:",
        "    raise RuntimeError(f'Gagal download vocab Emilia_ZH_EN_pinyin: {last_err}')",
        "print('vocab exists:', target.exists(), 'size:', target.stat().st_size if target.exists() else -1)",
    ])

    run_py(["-c", vocab_fetch_script], cwd=REPO_DIR)

run_cmd(["ls", "-lah", str(EMILIA_VOCAB_DIR)])


$ uv pip install --python /kaggle/temp/kcv-tts/.venv/bin/python -U huggingface_hub

$ uv run --no-sync --python /kaggle/temp/kcv-tts/.venv/bin/python python -c from pathlib import Path
import shutil
from huggingface_hub import hf_hub_download
target = Path(r'/kaggle/temp/kcv-tts/data/Emilia_ZH_EN_pinyin/vocab.txt')
target.parent.mkdir(parents=True, exist_ok=True)
candidates = [
    ('SWivid/F5-TTS', 'F5TTS_Base/vocab.txt'),
    ('SWivid/F5-TTS', 'F5TTS_v1_Base/vocab.txt'),
]
last_err = None
for repo_id, filename in candidates:
    try:
        src = Path(hf_hub_download(repo_id=repo_id, filename=filename))
        shutil.copy2(src, target)
        print(f'Downloaded vocab from {repo_id}/{filename} -> {target}')
        break
    except Exception as e:
        print(f'Gagal dari {repo_id}/{filename}: {e}')
        last_err = e
else:
    raise RuntimeError(f'Gagal download vocab Emilia_ZH_EN_pinyin: {last_err}')
print('vocab exists:', target.exists(), 'size:', target.stat().st_size if t

Using Python 3.11.15 environment at: /kaggle/temp/kcv-tts/.venv
Resolved 22 packages in 113ms
Audited 22 packages in 0.61ms


Downloaded vocab from SWivid/F5-TTS/F5TTS_Base/vocab.txt -> /kaggle/temp/kcv-tts/data/Emilia_ZH_EN_pinyin/vocab.txt
vocab exists: True size: 13800

$ ls -lah /kaggle/temp/kcv-tts/data/Emilia_ZH_EN_pinyin
total 24K
drwxr-xr-x 2 root root 4.0K Mar 29 17:56 .
drwxr-xr-x 3 root root 4.0K Mar 29 17:56 ..
-rw-r--r-- 1 root root  14K Mar 29 17:56 vocab.txt


CompletedProcess(args=['ls', '-lah', '/kaggle/temp/kcv-tts/data/Emilia_ZH_EN_pinyin'], returncode=0)

In [ ]:
# 7) Jalankan prepare_csv_wavs.py di notebook (Modal-aware vocab detect)
PREPARED_DATASET_DIR.mkdir(parents=True, exist_ok=True)

expected_vocab = REPO_DIR / "data" / "Emilia_ZH_EN_pinyin" / "vocab.txt"
if not expected_vocab.exists() or expected_vocab.stat().st_size == 0:
    data_root = REPO_DIR / "data"
    fallback = None
    for cand in data_root.glob("**/vocab.txt"):
        if cand.is_file() and cand.stat().st_size > 0:
            fallback = cand
            break

    if fallback is None:
        raise FileNotFoundError(
            f"vocab.txt tidak ditemukan untuk finetune prepare. Expected: {expected_vocab}"
        )

    expected_vocab.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(fallback, expected_vocab)
    print("Fallback vocab dipakai:", fallback, "->", expected_vocab)

print("Using pretrained vocab:", expected_vocab)

# Self-heal jika setup cell tidak dijalankan ulang setelah perubahan dependency profile.
try:
    run_py(["-c", "import f5_tts; print('f5_tts import ok')"], cwd=REPO_DIR)
except subprocess.CalledProcessError:
    print("f5_tts belum terinstall di venv, install editable package...")
    run_cmd([
        "uv",
        "pip",
        "install",
        "--python",
        str(VENV_PY),
        "-e",
        str(REPO_DIR),
    ], cwd=REPO_DIR)

run_py([
    "src/f5_tts/train/datasets/prepare_csv_wavs.py",
    str(MERGED_CSV),
    str(PREPARED_DATASET_DIR),
    "--workers",
    str(H100_NUM_WORKERS),
], cwd=REPO_DIR)

run_cmd(["ls", "-lah", str(PREPARED_DATASET_DIR)])

Using pretrained vocab: /kaggle/temp/kcv-tts/data/Emilia_ZH_EN_pinyin/vocab.txt

$ uv run --no-sync --python /kaggle/temp/kcv-tts/.venv/bin/python python -c import f5_tts; print('f5_tts import ok')
f5_tts belum terinstall di venv, install editable package...

$ uv pip install --python /kaggle/temp/kcv-tts/.venv/bin/python -e /kaggle/temp/kcv-tts


Traceback (most recent call last):
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'f5_tts'
Resolved 161 packages in 1.78s
Prepared 33 packages in 1.54s
Uninstalled 2 packages in 5ms
Installed 35 packages in 90ms
 + aiofiles==24.1.0
 + brotli==1.2.0
 + contourpy==1.3.3
 + cycler==0.12.1
 + encodec==0.1.1
 + f5-tts==1.1.18 (from file:///kaggle/temp/kcv-tts)
 + fastapi==0.135.2
 + ffmpy==1.0.0
 + fonttools==4.62.1
 - fsspec==2026.3.0
 + fsspec==2026.2.0
 + gradio==6.10.0
 + gradio-client==2.4.0
 + groovy==0.1.2
 + hf-gradio==0.3.0
 + kiwisolver==1.5.0
 + matplotlib==3.10.8
 + orjson==3.11.7
 + pydub==0.25.1
 + pyparsing==3.3.2
 + pypinyin==0.55.0
 + python-multipart==0.0.22
 + pytz==2026.1.post1
 - rich==14.3.3
 + rich==13.9.4
 + rjieba==0.2.0
 + safehttpx==0.1.7
 + semantic-version==2.10.0
 + starlette==0.52.1
 + tomli==2.4.1
 + tomlkit==0.13.3
 + torchcodec==0.11.0
 + torchdiffeq==0.2.5
 + transformers-stream-generator==0.0.5
 + unidecode==1.4.0
 + uvicorn==


$ uv run --no-sync --python /kaggle/temp/kcv-tts/.venv/bin/python python src/f5_tts/train/datasets/prepare_csv_wavs.py /kaggle/temp/kcv-tts/data/metadata_merged.csv /kaggle/temp/kcv-tts/data/datasetku_pinyin --workers 8

Processing 4972 audio files using 8 workers...


Writing to raw.arrow ...: 100%|██████████| 4972/4972 [00:00<00:00, 121309.53it/s]



Saving to /kaggle/temp/kcv-tts/data/datasetku_pinyin ...

For datasetku_pinyin, sample count: 4972
For datasetku_pinyin, vocab size is: 68
For datasetku_pinyin, total 5.11 hours

$ ls -lah /kaggle/temp/kcv-tts/data/datasetku_pinyin
total 2.0M
drwxr-xr-x 2 root root 4.0K Mar 29 17:56 .
drwxr-xr-x 4 root root 4.0K Mar 29 17:56 ..
-rw-r--r-- 1 root root  94K Mar 29 17:56 duration.json
-rw-r--r-- 1 root root 1.9M Mar 29 17:56 raw.arrow
-rw-r--r-- 1 root root  14K Mar 29 17:56 vocab.txt


CompletedProcess(args=['ls', '-lah', '/kaggle/temp/kcv-tts/data/datasetku_pinyin'], returncode=0)

In [12]:
# 8) W&B hardcoded login + sanity logging
env = os.environ.copy()
env["WANDB_API_KEY"] = WANDB_API_KEY
env["WANDB_ENTITY"] = WANDB_ENTITY
env["WANDB_PROJECT"] = WANDB_PROJECT

wandb_smoke = """
import os
import random
import wandb

api_key = os.environ["WANDB_API_KEY"]
entity = os.environ["WANDB_ENTITY"]
project = os.environ.get("WANDB_PROJECT", "kaceve")

wandb.login(key=api_key)
run = wandb.init(
    entity=entity,
    project=project,
    config={
        "learning_rate": 0.02,
        "architecture": "CNN",
        "dataset": "CIFAR-100",
        "epochs": 10,
    },
)
epochs = 10
offset = random.random() / 5
for epoch in range(2, epochs):
    acc = 1 - 2**-epoch - random.random() / epoch - offset
    loss = 2**-epoch + random.random() / epoch + offset
    run.log({"acc": acc, "loss": loss})
run.finish()
print("wandb sanity done")
""".strip()

run_py(["-c", wandb_smoke], cwd=REPO_DIR, env=env)


$ uv run --no-sync --python /kaggle/temp/kcv-tts/.venv/bin/python python -c import os
import random
import wandb

api_key = os.environ["WANDB_API_KEY"]
entity = os.environ["WANDB_ENTITY"]
project = os.environ.get("WANDB_PROJECT", "kaceve")

wandb.login(key=api_key)
run = wandb.init(
    entity=entity,
    project=project,
    config={
        "learning_rate": 0.02,
        "architecture": "CNN",
        "dataset": "CIFAR-100",
        "epochs": 10,
    },
)
epochs = 10
offset = random.random() / 5
for epoch in range(2, epochs):
    acc = 1 - 2**-epoch - random.random() / epoch - offset
    loss = 2**-epoch + random.random() / epoch + offset
    run.log({"acc": acc, "loss": loss})
run.finish()
print("wandb sanity done")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: haidarmuhammaddzaky (haidarmuhammaddzaky-institut-teknologi-sepuluh-nopember) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/temp/kcv-tts/wandb/run-20260329_175706-ya2v7y17
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run denim-sound-4
wandb: ⭐️ View project at https://wandb.ai/haidarmuhammaddzaky-institut-teknologi-sepuluh-nopember/kaceve
wandb: 🚀 View run at https://wandb.ai/haidarmuhammaddzaky-institut-teknologi-sep

wandb sanity done


wandb: 
wandb: Run history:
wandb:  acc ▁▅▇▇▇█▇▆
wandb: loss █▃▄▃▁▁▁▁
wandb: 
wandb: Run summary:
wandb:  acc 0.79451
wandb: loss 0.16708
wandb: 
wandb: 🚀 View run denim-sound-4 at: https://wandb.ai/haidarmuhammaddzaky-institut-teknologi-sepuluh-nopember/kaceve/runs/ya2v7y17
wandb: ⭐️ View project at: https://wandb.ai/haidarmuhammaddzaky-institut-teknologi-sepuluh-nopember/kaceve
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260329_175706-ya2v7y17/logs


CompletedProcess(args=['uv', 'run', '--no-sync', '--python', '/kaggle/temp/kcv-tts/.venv/bin/python', 'python', '-c', 'import os\nimport random\nimport wandb\n\napi_key = os.environ["WANDB_API_KEY"]\nentity = os.environ["WANDB_ENTITY"]\nproject = os.environ.get("WANDB_PROJECT", "kaceve")\n\nwandb.login(key=api_key)\nrun = wandb.init(\n    entity=entity,\n    project=project,\n    config={\n        "learning_rate": 0.02,\n        "architecture": "CNN",\n        "dataset": "CIFAR-100",\n        "epochs": 10,\n    },\n)\nepochs = 10\noffset = random.random() / 5\nfor epoch in range(2, epochs):\n    acc = 1 - 2**-epoch - random.random() / epoch - offset\n    loss = 2**-epoch + random.random() / epoch + offset\n    run.log({"acc": acc, "loss": loss})\nrun.finish()\nprint("wandb sanity done")'], returncode=0)

In [ ]:
# 9) Persiapan runtime training + mamba (tanpa menjalankan training)
env = os.environ.copy()
env["WANDB_API_KEY"] = WANDB_API_KEY
env["WANDB_ENTITY"] = WANDB_ENTITY
env["WANDB_PROJECT"] = WANDB_PROJECT

CKPT_ROOT_WORKING = WORKDIR / "ckpts"
DISTILL_TAG = "distill_final_datasetku"
FULL_TAG = "full_final_datasetku"

distill_dir_abs = CKPT_ROOT_WORKING / DISTILL_TAG
full_dir_abs = CKPT_ROOT_WORKING / FULL_TAG
distill_dir_abs.mkdir(parents=True, exist_ok=True)
full_dir_abs.mkdir(parents=True, exist_ok=True)

# train.py menyimpan checkpoint ke repo_root/ckpts.save_dir, jadi dari REPO_DIR cukup naik 1 level ke WORKDIR.
distill_save_dir_rel = f"../ckpts/{DISTILL_TAG}"
full_save_dir_rel = f"../ckpts/{FULL_TAG}"

no_periodic_ckpt_overrides = [
    "ckpts.save_per_updates=999999999",
    "ckpts.last_per_updates=999999999",
    "ckpts.keep_last_n_checkpoints=0",
    "ckpts.log_samples=False",
]

MAMBA_PROBE_TIMEOUT_SEC = 90
MAMBA_POST_REPAIR_TIMEOUT_SEC = 240

def _build_runtime_env(base_env):
    runtime_env = base_env.copy()
    site_pkgs_candidates = sorted((VENV_DIR / "lib").glob("python*/site-packages"))
    if not site_pkgs_candidates:
        return runtime_env

    sp = site_pkgs_candidates[-1]
    cuda_lib_rels = [
        "nvidia/cublas/lib",
        "nvidia/cuda_runtime/lib",
        "nvidia/cudnn/lib",
        "nvidia/cufft/lib",
        "nvidia/curand/lib",
        "nvidia/cusolver/lib",
        "nvidia/cusparse/lib",
        "nvidia/nccl/lib",
        "nvidia/nvjitlink/lib",
    ]
    cuda_libs = [str(sp / rel) for rel in cuda_lib_rels if (sp / rel).exists()]
    if cuda_libs:
        current = runtime_env.get("LD_LIBRARY_PATH", "")
        runtime_env["LD_LIBRARY_PATH"] = ":".join(cuda_libs + ([current] if current else []))
        print("LD_LIBRARY_PATH prepared for CUDA libs in venv.")

    runtime_env.setdefault("PYTHONFAULTHANDLER", "1")
    runtime_env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    return runtime_env

runtime_env = _build_runtime_env(env)

def run_py_nosync(args, cwd=None, timeout=None):
    return run_cmd(
        ["uv", "run", "--no-sync", "--python", str(VENV_PY), "python", *args],
        cwd=cwd,
        env=runtime_env,
        timeout=timeout,
    )

def _probe_mamba(timeout_sec=MAMBA_PROBE_TIMEOUT_SEC) -> None:
    probe_script = "\n".join([
        "print('probe: import torch...', flush=True)",
        "import torch",
        "print('probe: torch =', torch.__version__, 'cuda =', torch.version.cuda, flush=True)",
        "print('probe: import mamba_ssm...', flush=True)",
        "import mamba_ssm",
        "print('probe: import selective_scan_cuda...', flush=True)",
        "import selective_scan_cuda",
        "print('mamba probe ok', flush=True)",
    ])
    run_py_nosync(["-u", "-X", "faulthandler", "-c", probe_script], cwd=REPO_DIR, timeout=timeout_sec)

def _ensure_torch_compat() -> None:
    check_script = "\n".join([
        "import torch",
        "print('torch =', torch.__version__, 'cuda =', torch.version.cuda)",
        "ok = torch.__version__.startswith('2.8.0') and str(torch.version.cuda).startswith('12.8')",
        "raise SystemExit(0 if ok else 1)",
    ])
    try:
        run_py_nosync(["-c", check_script], cwd=REPO_DIR, timeout=60)
        return
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
        print("Torch stack tidak cocok, paksa reinstall ke torch 2.8.0+cu128.")

    run_cmd([
        "uv",
        "pip",
        "install",
        "--python",
        str(VENV_PY),
        "--index-url",
        "https://download.pytorch.org/whl/cu128",
        "--extra-index-url",
        "https://pypi.org/simple",
        "--index-strategy",
        "unsafe-best-match",
        "--force-reinstall",
        "--no-cache-dir",
        "torch==2.8.0+cu128",
        "torchvision==0.23.0+cu128",
        "torchaudio==2.8.0+cu128",
        "nvidia-nccl-cu12==2.27.3",
        "nvidia-nvjitlink-cu12==12.8.93",
    ], cwd=REPO_DIR)

    run_py_nosync(["-c", check_script], cwd=REPO_DIR, timeout=60)

def _ensure_python_headers() -> None:
    py_mm = subprocess.check_output(
        [str(VENV_PY), "-c", "import sys; print(f'{sys.version_info.major}.{sys.version_info.minor}')"],
        text=True,
    ).strip()
    py_header = Path(f"/usr/include/python{py_mm}/Python.h")
    if py_header.exists():
        return

    print(f"Python headers tidak ditemukan ({py_header}), install python{py_mm}-dev...")
    run_cmd(["apt-get", "update", "-y"])
    run_cmd(["apt-get", "install", "-y", f"python{py_mm}-dev", "build-essential"])

    if not py_header.exists():
        raise FileNotFoundError(f"Python.h tetap tidak ditemukan di {py_header}")

def _repair_mamba() -> None:
    print("Repair mamba dimulai...")
    _ensure_torch_compat()
    _ensure_python_headers()

    run_cmd(
        [str(VENV_PY), "-m", "pip", "install", "--upgrade", "pip", "wheel", "ninja", "setuptools<82"],
        cwd=REPO_DIR,
        env=runtime_env,
    )

    wheel_env = runtime_env.copy()
    wheel_env["TORCH_CUDA_ARCH_LIST"] = "9.0"
    wheel_env["MAX_JOBS"] = "8"

    wheel_cmd = [
        "uv",
        "pip",
        "install",
        "--python",
        str(VENV_PY),
        "--index-url",
        "https://pypi.org/simple",
        "--extra-index-url",
        "https://download.pytorch.org/whl/cu128",
        "--index-strategy",
        "unsafe-best-match",
        "--force-reinstall",
        "--no-cache-dir",
        "--prefer-binary",
        "--no-build-isolation",
        "--no-deps",
        "causal-conv1d",
        "mamba-ssm",
    ]

    try:
        run_cmd(wheel_cmd, cwd=REPO_DIR, env=wheel_env)
        _probe_mamba(timeout_sec=120)
        print("Mamba wheel kompatibel.")
        return
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
        print("Wheel mamba gagal/timeout, lanjut build from source...")

    build_env = wheel_env.copy()
    build_env["MAMBA_FORCE_BUILD"] = "TRUE"
    build_env["CAUSAL_CONV1D_FORCE_BUILD"] = "TRUE"

    run_cmd(
        [
            str(VENV_PY),
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "--no-build-isolation",
            "--force-reinstall",
            "--no-binary",
            ":all:",
            "--no-deps",
            "causal-conv1d",
        ],
        cwd=REPO_DIR,
        env=build_env,
    )
    run_cmd(
        [
            str(VENV_PY),
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "--no-build-isolation",
            "--force-reinstall",
            "--no-binary",
            ":all:",
            "--no-deps",
            "mamba-ssm",
        ],
        cwd=REPO_DIR,
        env=build_env,
    )

    _probe_mamba(timeout_sec=MAMBA_POST_REPAIR_TIMEOUT_SEC)
    print("Mamba berhasil dibangun dari source.")

try:
    _probe_mamba()
    print("mamba_mode: enabled (fast-path)")
except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
    print("Probe mamba gagal/timeout. Menjalankan repair...")
    _repair_mamba()
    _probe_mamba(timeout_sec=MAMBA_POST_REPAIR_TIMEOUT_SEC)
    print("mamba_mode: enabled (after repair)")

print("Persiapan selesai. Lanjut ke cell distill.")

LD_LIBRARY_PATH prepared for CUDA libs in venv.

$ uv run --no-sync --python /kaggle/temp/kcv-tts/.venv/bin/python python -u -X faulthandler -c print('probe: import torch...', flush=True)
import torch
print('probe: torch =', torch.__version__, 'cuda =', torch.version.cuda, flush=True)
print('probe: import mamba_ssm...', flush=True)
import mamba_ssm
print('probe: import selective_scan_cuda...', flush=True)
import selective_scan_cuda
print('mamba probe ok', flush=True)
(timeout=90s)
probe: import torch...
probe: torch = 2.8.0+cu128 cuda = 12.8
probe: import mamba_ssm...
probe: import selective_scan_cuda...
mamba probe ok
mamba_mode: enabled (fast-path)
Persiapan selesai. Lanjut ke cell distill.


In [ ]:
# 10) Distill phase (final checkpoint only, 1x H100)
distill_run_name = "F5TTS_HYBRID_DISTILL_5K_datasetku_modal_1xH100"

attn_backend_overrides = []
if H100_USE_FLASH_ATTN:
    try:
        run_py_nosync(["-c", "import flash_attn; print('flash_attn available')"], cwd=REPO_DIR, timeout=60)
        attn_backend_overrides = ["model.arch.attn_backend=flash_attn"]
        print("Using flash_attn backend for H100.")
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
        print("flash_attn tidak tersedia. Fallback ke backend torch.")

distill_cmd = [
    "uv",
    "run",
    "--no-sync",
    "--python",
    str(VENV_PY),
    "accelerate",
    "launch",
    "--num_processes=1",
    f"--mixed_precision={H100_MIXED_PRECISION}",
    "src/f5_tts/train/train.py",
    "--config-name",
    "F5TTS_HYBRID_DISTILL_5K.yaml",
    "datasets.name=datasetku",
    f"model.cfm_experiment.teacher_ckpt_path={TEACHER_CKPT}",
    f"datasets.batch_size_per_gpu={H100_DISTILL_BATCH_FRAMES}",
    f"datasets.max_samples={H100_MAX_SAMPLES}",
    f"datasets.num_workers={H100_NUM_WORKERS}",
    "optim.epochs=1",
    "optim.num_warmup_updates=5000",
    "model.cfm_experiment.lambda_distill_out=0.1",
    "ckpts.logger=wandb",
    f"ckpts.wandb_project={WANDB_PROJECT}",
    f"ckpts.wandb_run_name={distill_run_name}",
    f"ckpts.save_dir={distill_save_dir_rel}",
    *attn_backend_overrides,
    *no_periodic_ckpt_overrides,
]

run_cmd(distill_cmd, cwd=REPO_DIR, env=runtime_env)
run_cmd(["ls", "-lah", str(distill_dir_abs)])

distill_last = distill_dir_abs / "model_last.pt"
if not distill_last.exists():
    raise FileNotFoundError(f"Distill final checkpoint tidak ditemukan: {distill_last}")

# full training BUKAN dari nol: partial init dari parent checkpoint yang sama dengan teacher distill
if not TEACHER_CKPT.exists():
    raise FileNotFoundError(f"Parent checkpoint tidak ditemukan: {TEACHER_CKPT}")

pretrained_for_full = full_dir_abs / "pretrained_indo_v2.pt"
shutil.copy2(TEACHER_CKPT, pretrained_for_full)

print("Distill final checkpoint:", distill_last)
print("Prepared full-training partial init checkpoint (same parent as distill):", pretrained_for_full)
print("Distill selesai. Lanjut ke cell full training.")


$ uv run --no-sync --python /kaggle/temp/kcv-tts/.venv/bin/python accelerate launch --num_processes=2 --mixed_precision=fp16 src/f5_tts/train/train.py --config-name F5TTS_HYBRID_DISTILL_5K.yaml datasets.name=datasetku model.cfm_experiment.teacher_ckpt_path=/kaggle/temp/kcv-tts/ckpts/hf/Eempostor_F5-TTS-INDO-FINETUNE-V2/f5_tts_indo_v2.pt datasets.batch_size_per_gpu=768 datasets.max_samples=1 datasets.num_workers=4 optim.epochs=1 optim.num_warmup_updates=500 ckpts.logger=wandb ckpts.wandb_project=kaceve ckpts.wandb_run_name=F5TTS_HYBRID_DISTILL_5K_datasetku_kaggle_2xT4 ckpts.save_dir=../../working/ckpts/distill_final_datasetku ckpts.save_per_updates=999999999 ckpts.last_per_updates=999999999 ckpts.keep_last_n_checkpoints=0 ckpts.log_samples=False


The following values were not passed to `accelerate launch` and had defaults used instead:
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.


[Distill Teacher] backbone=DiT checkpoint=/kaggle/temp/kcv-tts/ckpts/hf/Eempostor_F5-TTS-INDO-FINETUNE-V2/f5_tts_indo_v2.pt loaded=364/364 missing=0 unexpected=2 shape_mismatch=0
[Distill Teacher] backbone=DiT checkpoint=/kaggle/temp/kcv-tts/ckpts/hf/Eempostor_F5-TTS-INDO-FINETUNE-V2/f5_tts_indo_v2.pt loaded=364/364 missing=0 unexpected=2 shape_mismatch=0


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: haidarmuhammaddzaky (haidarmuhammaddzaky-institut-teknologi-sepuluh-nopember) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run pnbliuol
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/temp/kcv-tts/wandb/run-20260329_175749-pnbliuol
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run F5TTS_HYBRID_DISTILL_5K_datasetku_kaggle_2xT4
wandb: ⭐️ View project at https://wandb.ai/haidarmuhammaddzaky-institut-teknologi-sepuluh-nopember/kaceve
wandb: 🚀 View run at https://wandb.ai/haidarmuhammaddzaky-institut-teknologi-sepuluh-nopember/kaceve/runs/pnbliuol


Using logger: wandb
Loading dataset ...
Loading dataset ...
[Training Config Summary] epochs=1 lr=2e-05 warmup_updates=500 batch_size_per_gpu=768 batch_size_type=frame max_samples=1 grad_accumulation_steps=1 max_grad_norm=1.0
[CFM Experimental Summary] backbone=HybridDiT use_mamba=True mamba_layers=[10, 11] use_distill=True lambda_distill_out=0.050000 lambda_distill_hidden=0.000000 use_ctc=False lambda_ctc=0.000000 use_accent_adv=False lambda_adv=0.000000


Sorting with sampler... if slow, check whether dataset is provided with duration: 100%|██████████| 4972/4972 [00:00<00:00, 1035970.17it/s]
Sorting with sampler... if slow, check whether dataset is provided with duration: 100%|██████████| 4972/4972 [00:00<00:00, 1406778.16it/s]
Creating dynamic batches with 768 audio frames per gpu: 100%|██████████| 4972/4972 [00:00<00:00, 1835423.30it/s]
Creating dynamic batches with 768 audio frames per gpu: 100%|██████████| 4972/4972 [00:00<00:00, 1922743.82it/s]
Epoch 1/1:   0%|          | 0/2478 [00:00<?, ?update/s]/kaggle/temp/kcv-tts/.venv/lib/python3.11/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable

Saved last checkpoint at update 2478


wandb: updating run metadata
wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: uploading output.log; uploading config.yaml
wandb: uploading history steps 2477-2477, summary, console lines 7-7
wandb: 
wandb: Run history:
wandb:           grad_norm ▆▄▄▄▆▂▅▄▄▅▃▃▃▃▃▃▂▂▃▄▃▂▃▂▃▃▂█▂█▂▂▁▂▁▁▂▁▆▁
wandb:                loss ▅█▅▃▂▄▁▄▂▂▄▃▁▂▃▃▁▁▁▅▁▁▁▁▁▂▁▁▁▁▁▁▁▂▂▁▁▁▁▅
wandb:            loss_adv ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:            loss_ctc ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: loss_distill_hidden ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:    loss_distill_out ▅▇█▃▂▃▂▂▁▂▁▁▁▂▂▂▁▁▂▁▁▂▁▁▁▁▁▂▁▁▁▂▁▁▁▂▁▁▁▁
wandb:           loss_flow ▇▅▄█▃▂▂▂▃▂▆▂▁▁▂▂▂▂▂▁▂▂▁▁▂▁▂▁▃▁▁▁▁▁▁▃▁▁▁▁
wandb:          loss_total █▇▃▃▆▂▄▅▂▄▁▃▂▂▂▃▁▂▂▅▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▂▁▁▁
wandb:                  lr ▂▃▃▄▇█████▇▇▇▇▇▇▇▆▆▆▆▆▆▅▄▄▄▃▃▃▂▂▂▂▂▂▂▁▁▁
wandb: 
wandb: Run summary:
wandb:           grad_norm 7.24239
wandb:                loss 1.57475
wandb:            loss_adv 0
wan


$ ls -lah /kaggle/working/ckpts/distill_final_datasetku
total 7.9G
drwxr-xr-x 2 root root 4.0K Mar 29 18:26 .
drwxr-xr-x 4 root root 4.0K Mar 29 17:57 ..
-rw-r--r-- 1 root root 7.9G Mar 29 18:26 model_last.pt
Distill final checkpoint: /kaggle/working/ckpts/distill_final_datasetku/model_last.pt
Prepared full-training partial init checkpoint (same parent as distill): /kaggle/working/ckpts/full_final_datasetku/pretrained_indo_v2.pt
Distill selesai. Lanjut ke cell full training.


In [16]:
# 10) HOTFIX runtime (sementara): patch checkpoint-compat di trainer.py
from pathlib import Path

trainer_path = REPO_DIR / "src/f5_tts/model/trainer.py"
src = trainer_path.read_text(encoding="utf-8")

if "Checkpoint loaded in compatibility mode; update reset to 0" in src:
    print("Hotfix sudah ada di trainer.py")
else:
    # Gunakan string literal dengan indent asli supaya patch aman untuk runtime process lain.
    old_ema_block = (
        "        if self.is_main:\n"
        "            self.ema_model.load_state_dict(checkpoint[\"ema_model_state_dict\"])\n"
    )
    new_ema_block = (
        "        ema_loaded_strict = True\n"
        "        if self.is_main:\n"
        "            try:\n"
        "                self.ema_model.load_state_dict(checkpoint[\"ema_model_state_dict\"])\n"
        "            except RuntimeError as exc:\n"
        "                ema_loaded_strict = False\n"
        "                print(f\"Strict EMA checkpoint load failed, retrying non-strict for compatibility: {exc}\")\n"
        "                self.ema_model.load_state_dict(checkpoint[\"ema_model_state_dict\"], strict=False)\n"
    )

    old_resume_block = (
        "            try:\n"
        "                self.accelerator.unwrap_model(self.model).load_state_dict(checkpoint[\"model_state_dict\"])\n"
        "            except RuntimeError as exc:\n"
        "                print(f\"Strict checkpoint load failed, retrying non-strict for compatibility: {exc}\")\n"
        "                self.accelerator.unwrap_model(self.model).load_state_dict(checkpoint[\"model_state_dict\"], strict=False)\n"
        "            self.optimizer.load_state_dict(checkpoint[\"optimizer_state_dict\"])\n"
        "            if self.scheduler:\n"
        "                self.scheduler.load_state_dict(checkpoint[\"scheduler_state_dict\"])\n"
        "            update = checkpoint[\"update\"]\n"
    )

    new_resume_block = (
        "            model_loaded_strict = True\n"
        "            try:\n"
        "                self.accelerator.unwrap_model(self.model).load_state_dict(checkpoint[\"model_state_dict\"])\n"
        "            except RuntimeError as exc:\n"
        "                model_loaded_strict = False\n"
        "                print(f\"Strict checkpoint load failed, retrying non-strict for compatibility: {exc}\")\n"
        "                self.accelerator.unwrap_model(self.model).load_state_dict(checkpoint[\"model_state_dict\"], strict=False)\n"
        "\n"
        "            optimizer_loaded = True\n"
        "            try:\n"
        "                self.optimizer.load_state_dict(checkpoint[\"optimizer_state_dict\"])\n"
        "            except Exception as exc:\n"
        "                optimizer_loaded = False\n"
        "                print(f\"Optimizer checkpoint load failed, using fresh optimizer state: {exc}\")\n"
        "\n"
        "            scheduler_loaded = True\n"
        "            if self.scheduler:\n"
        "                try:\n"
        "                    self.scheduler.load_state_dict(checkpoint[\"scheduler_state_dict\"])\n"
        "                except Exception as exc:\n"
        "                    scheduler_loaded = False\n"
        "                    print(f\"Scheduler checkpoint load failed, using fresh scheduler state: {exc}\")\n"
        "\n"
        "            if model_loaded_strict and optimizer_loaded and scheduler_loaded and ema_loaded_strict:\n"
        "                update = checkpoint[\"update\"]\n"
        "            else:\n"
        "                update = 0\n"
        "                print(\n"
        "                    \"Checkpoint loaded in compatibility mode; update reset to 0 \"\n"
        "                    \"to avoid resuming with incompatible optimizer/scheduler state.\"\n"
        "                )\n"
    )

    missing = []
    if old_ema_block in src:
        src = src.replace(old_ema_block, new_ema_block, 1)
    elif new_ema_block in src:
        pass
    else:
        missing.append("EMA load block")

    if old_resume_block in src:
        src = src.replace(old_resume_block, new_resume_block, 1)
    elif new_resume_block in src:
        pass
    else:
        missing.append("resume block")

    if missing:
        raise RuntimeError(
            "Gagal patch trainer.py, blok tidak ditemukan: " + ", ".join(missing)
        )

    trainer_path.write_text(src, encoding="utf-8")
    print("Hotfix berhasil diterapkan:", trainer_path)

run_py(["-m", "py_compile", "src/f5_tts/model/trainer.py"], cwd=REPO_DIR)
print("trainer.py syntax OK")
run_py([
    "-c",
    "from f5_tts.model.trainer import Trainer; print('trainer import OK after hotfix')",
], cwd=REPO_DIR)

Hotfix berhasil diterapkan: /kaggle/temp/kcv-tts/src/f5_tts/model/trainer.py

$ uv run --no-sync --python /kaggle/temp/kcv-tts/.venv/bin/python python -m py_compile src/f5_tts/model/trainer.py
trainer.py syntax OK

$ uv run --no-sync --python /kaggle/temp/kcv-tts/.venv/bin/python python -c from f5_tts.model.trainer import Trainer; print('trainer import OK after hotfix')
trainer import OK after hotfix


CompletedProcess(args=['uv', 'run', '--no-sync', '--python', '/kaggle/temp/kcv-tts/.venv/bin/python', 'python', '-c', "from f5_tts.model.trainer import Trainer; print('trainer import OK after hotfix')"], returncode=0)

In [ ]:
# 11) Full training phase (final checkpoint only, 1x H100)
pretrained_for_full = full_dir_abs / "pretrained_indo_v2.pt"
if not pretrained_for_full.exists():
    raise FileNotFoundError(
        f"Partial init checkpoint untuk full training belum ada: {pretrained_for_full}. Jalankan cell distill dulu."
    )

full_run_name = "F5TTS_EarlyBiMamba_v1_datasetku_modal_1xH100"

attn_backend_overrides = []
if H100_USE_FLASH_ATTN:
    try:
        run_py_nosync(["-c", "import flash_attn; print('flash_attn available')"], cwd=REPO_DIR, timeout=60)
        attn_backend_overrides = ["model.arch.attn_backend=flash_attn"]
        print("Using flash_attn backend for H100.")
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
        print("flash_attn tidak tersedia. Fallback ke backend torch.")

full_cmd = [
    "uv",
    "run",
    "--no-sync",
    "--python",
    str(VENV_PY),
    "accelerate",
    "launch",
    "--num_processes=1",
    f"--mixed_precision={H100_MIXED_PRECISION}",
    "src/f5_tts/train/train.py",
    "--config-name",
    "F5TTS_EarlyBiMamba_v1.yaml",
    "datasets.name=datasetku",
    f"datasets.batch_size_per_gpu={H100_FULL_BATCH_FRAMES}",
    f"datasets.max_samples={H100_MAX_SAMPLES}",
    f"datasets.num_workers={H100_NUM_WORKERS}",
    "optim.epochs=40",
    "optim.num_warmup_updates=20000",
    "ckpts.logger=wandb",
    f"ckpts.wandb_project={WANDB_PROJECT}",
    f"ckpts.wandb_run_name={full_run_name}",
    f"ckpts.save_dir={full_save_dir_rel}",
    *attn_backend_overrides,
    *no_periodic_ckpt_overrides,
]

run_cmd(full_cmd, cwd=REPO_DIR, env=runtime_env)
run_cmd(["ls", "-lah", str(full_dir_abs)])

full_last = full_dir_abs / "model_last.pt"
if not full_last.exists():
    raise FileNotFoundError(f"Full-training final checkpoint tidak ditemukan: {full_last}")

distill_last = distill_dir_abs / "model_last.pt"
if not distill_last.exists():
    raise FileNotFoundError(f"Distill final checkpoint tidak ditemukan: {distill_last}")

print("\nFinal checkpoints:")
print("- Distill final:", distill_last)
print("- Full final:", full_last)


$ uv run --no-sync --python /kaggle/temp/kcv-tts/.venv/bin/python accelerate launch --num_processes=2 --mixed_precision=fp16 src/f5_tts/train/train.py --config-name F5TTS_EarlyBiMamba_v1.yaml datasets.name=datasetku datasets.batch_size_per_gpu=768 datasets.max_samples=1 datasets.num_workers=4 optim.epochs=1 optim.num_warmup_updates=500 ckpts.logger=wandb ckpts.wandb_project=kaceve ckpts.wandb_run_name=F5TTS_EarlyBiMamba_v1_datasetku_kaggle_2xT4 ckpts.save_dir=../../working/ckpts/full_final_datasetku ckpts.save_per_updates=999999999 ckpts.last_per_updates=999999999 ckpts.keep_last_n_checkpoints=0 ckpts.log_samples=False


The following values were not passed to `accelerate launch` and had defaults used instead:
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: haidarmuhammaddzaky (haidarmuhammaddzaky-institut-teknologi-sepuluh-nopember) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run 05dsz4sp
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/temp/kcv-tts/wandb/run-20260329_184720-05dsz4sp
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run F5TTS_EarlyBiMamba_v1_datasetku_kaggle_2xT4
wandb: ⭐️ View project at https://wandb.ai/ha

Using logger: wandb
Loading dataset ...
Loading dataset ...
[Training Config Summary] epochs=1 lr=7.5e-05 warmup_updates=500 batch_size_per_gpu=768 batch_size_type=frame max_samples=1 grad_accumulation_steps=1 max_grad_norm=1.0
[CFM Experimental Summary] backbone=HybridDiT use_mamba=True mamba_layers=[0, 1, 2, 3, 4, 5, 6, 7] use_distill=False lambda_distill_out=0.000000 lambda_distill_hidden=0.000000 use_ctc=False lambda_ctc=0.000000 use_accent_adv=False lambda_adv=0.000000


Sorting with sampler... if slow, check whether dataset is provided with duration: 100%|██████████| 4972/4972 [00:00<00:00, 1266724.14it/s]
Sorting with sampler... if slow, check whether dataset is provided with duration: 100%|██████████| 4972/4972 [00:00<00:00, 1672742.40it/s]
Creating dynamic batches with 768 audio frames per gpu: 100%|██████████| 4972/4972 [00:00<00:00, 1562454.45it/s]
Creating dynamic batches with 768 audio frames per gpu: 100%|██████████| 4972/4972 [00:00<00:00, 1605271.30it/s]


Strict EMA checkpoint load failed, retrying non-strict for compatibility: Error(s) in loading state_dict for EMA:
	Missing key(s) in state_dict: "ema_model.transformer.transformer_blocks.0.mixer.pos_scale", "ema_model.transformer.transformer_blocks.0.mixer.fwd.A_log", "ema_model.transformer.transformer_blocks.0.mixer.fwd.D", "ema_model.transformer.transformer_blocks.0.mixer.fwd.in_proj.weight", "ema_model.transformer.transformer_blocks.0.mixer.fwd.conv1d.weight", "ema_model.transformer.transformer_blocks.0.mixer.fwd.conv1d.bias", "ema_model.transformer.transformer_blocks.0.mixer.fwd.x_proj.weight", "ema_model.transformer.transformer_blocks.0.mixer.fwd.dt_proj.weight", "ema_model.transformer.transformer_blocks.0.mixer.fwd.dt_proj.bias", "ema_model.transformer.transformer_blocks.0.mixer.fwd.out_proj.weight", "ema_model.transformer.transformer_blocks.0.mixer.bwd.A_log", "ema_model.transformer.transformer_blocks.0.mixer.bwd.D", "ema_model.transformer.transformer_blocks.0.mixer.bwd.in_proj.

/kaggle/temp/kcv-tts/.venv/lib/python3.11/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/kaggle/temp/kcv-tts/.venv/lib/python3.11/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchco

[Checkpoint Audit]
  total_params_student=431,665,292 total_params_loaded_exact=303,509,636 total_params_new_or_missing=128,155,656 load_ratio_percent=70.31%
  missing_keys=160 (expected=160, suspicious=0)
  unexpected_keys=64 (expected=64, suspicious=0)
  missing_keys grouped by module:
  - transformer.transformer_blocks.0.mixer: 20
  - transformer.transformer_blocks.1.mixer: 20
  - transformer.transformer_blocks.2.mixer: 20
  - transformer.transformer_blocks.3.mixer: 20
  - transformer.transformer_blocks.4.mixer: 20
  - transformer.transformer_blocks.5.mixer: 20
  - transformer.transformer_blocks.6.mixer: 20
  - transformer.transformer_blocks.7.mixer: 20
  unexpected_keys grouped by module:
  - transformer.transformer_blocks.0.attn: 8
  - transformer.transformer_blocks.1.attn: 8
  - transformer.transformer_blocks.2.attn: 8
  - transformer.transformer_blocks.3.attn: 8
  - transformer.transformer_blocks.4.attn: 8
  - transformer.transformer_blocks.5.attn: 8
  - transformer.transformer_

Epoch 1/1:   0%|          | 0/2478 [00:00<?, ?update/s]/kaggle/temp/kcv-tts/.venv/lib/python3.11/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/kaggle/temp/kcv-tts/.venv/lib/python3.11/site-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Ple

KeyboardInterrupt: 

In [ ]:
# 12) Push final checkpoints ke Hugging Face Hub (new private repo)
if not distill_last.exists() or not full_last.exists():
    raise FileNotFoundError(
        "Checkpoint final belum lengkap. Jalankan cell full training dulu."
    )

run_cmd(["uv", "pip", "install", "--python", str(VENV_PY), "-U", "huggingface_hub"], cwd=REPO_DIR)

upload_env = os.environ.copy()
upload_env["HF_TOKEN"] = HF_TOKEN

upload_script = "\n".join([
    "import os",
    "from datetime import datetime, timezone",
    "from pathlib import Path",
    "from huggingface_hub import HfApi",
    f"distill_last = Path(r'{distill_last}')",
    f"full_last = Path(r'{full_last}')",
    f"workdir = Path(r'{WORKDIR}')",
    f"repo_prefix = {HF_OUTPUT_REPO_PREFIX!r}",
    f"repo_owner_override = {HF_OUTPUT_REPO_OWNER!r}",
    f"repo_id_override = {HF_OUTPUT_REPO_ID!r}",

    "if not distill_last.exists() or not full_last.exists():",
    "    raise FileNotFoundError('Checkpoint final belum lengkap. Jalankan cell full training dulu.')",

    "hf_token = os.environ['HF_TOKEN']",
    "api = HfApi(token=hf_token)",
    "whoami = api.whoami(token=hf_token)",
    "owner = repo_owner_override or whoami.get('name')",
    "if not owner:",
    "    raise RuntimeError('Tidak bisa menentukan owner Hugging Face. Set HF_OUTPUT_REPO_OWNER di environment.')",

    "timestamp = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')",
    "target_repo_id = repo_id_override or f'{owner}/{repo_prefix}-{timestamp}'",
    "print('Target HF repo:', target_repo_id)",
    "print('Mode: private + must be new (exist_ok=False)')",

    "api.create_repo(",
    "    repo_id=target_repo_id,",
    "    repo_type='model',",
    "    private=True,",
    "    exist_ok=False,",
    "    token=hf_token,",
    ")",

    "model_card = '\\n'.join([",
    "    '# KCV-TTS Modal H100 Checkpoints',",
    "    '',",
    "    'Repo ini dibuat otomatis dari notebook trainzeh4-workbanget-modal-h100.ipynb.',",
    "    f'- Distill checkpoint: {distill_last.name}',",
    "    f'- Full checkpoint: {full_last.name}',",
    "    '',",
    "    'Checkpoint disimpan private secara default.',",
    "])",
    "readme_path = workdir / 'README_modal_h100_upload.md'",
    "readme_path.write_text(model_card, encoding='utf-8')",

    "api.upload_file(",
    "    path_or_fileobj=str(distill_last),",
    "    path_in_repo='checkpoints/distill/model_last.pt',",
    "    repo_id=target_repo_id,",
    "    repo_type='model',",
    "    token=hf_token,",
    ")",
    "api.upload_file(",
    "    path_or_fileobj=str(full_last),",
    "    path_in_repo='checkpoints/full/model_last.pt',",
    "    repo_id=target_repo_id,",
    "    repo_type='model',",
    "    token=hf_token,",
    ")",
    "api.upload_file(",
    "    path_or_fileobj=str(readme_path),",
    "    path_in_repo='README.md',",
    "    repo_id=target_repo_id,",
    "    repo_type='model',",
    "    token=hf_token,",
    ")",

    "print('Upload selesai.')",
    "print('Repo URL:', f'https://huggingface.co/{target_repo_id}')",
    "print('Distill URL:', f'https://huggingface.co/{target_repo_id}/blob/main/checkpoints/distill/model_last.pt')",
    "print('Full URL:', f'https://huggingface.co/{target_repo_id}/blob/main/checkpoints/full/model_last.pt')",
])

run_py(["-c", upload_script], cwd=REPO_DIR, env=upload_env)

In [ ]:
# One-cell inference test untuk trainzeh4-workbanget-modal-a100-40gb
from pathlib import Path
from IPython.display import Audio, display
import csv
import os
import subprocess

ckpt_file = Path(full_last if "full_last" in globals() and Path(full_last).exists() else distill_last)
if not ckpt_file.exists():
    raise FileNotFoundError(f"Checkpoint tidak ditemukan: {ckpt_file}")

if not MERGED_CSV.exists():
    raise FileNotFoundError(f"MERGED_CSV tidak ditemukan: {MERGED_CSV}")

with open(MERGED_CSV, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f, delimiter="|")
    rows = list(reader)

if not rows:
    raise ValueError(f"metadata kosong: {MERGED_CSV}")


def resolve_audio_path(p: str) -> Path:
    p = str(p).strip()
    cand = Path(p)
    candidates = [
        cand,
        DATASET_ROOT / p,
        DATASET_DATA_DIR / p,
        REPO_DIR / p,
        WORKDIR / p,
    ]
    for c in candidates:
        if c.exists():
            return c.resolve()
    raise FileNotFoundError(f"Audio referensi tidak ketemu: {p}")


ref_row = next((r for r in rows if str(r.get("audio_file", "")).strip() and str(r.get("text", "")).strip()), None)
if ref_row is None:
    raise ValueError("Tidak ada baris valid di metadata_merged.csv")

ref_audio = resolve_audio_path(ref_row["audio_file"])
ref_text = str(ref_row["text"]).strip()
gen_text = "HALO"

vocab_file = REPO_DIR / "data" / "Emilia_ZH_EN_pinyin" / "vocab.txt"
if not vocab_file.exists():
    fallback_vocab = next((p for p in (REPO_DIR / "data").glob("**/vocab.txt") if p.is_file() and p.stat().st_size > 0), None)
    if fallback_vocab is None:
        raise FileNotFoundError("vocab.txt tidak ditemukan")
    vocab_file = fallback_vocab

out_wav = WORKDIR / "infer_trainzeh4_modal_a100_40gb_test.wav"

infer_script = f"""
from pathlib import Path
from f5_tts.api import F5TTS
import torch

ckpt_file = Path(r"{str(ckpt_file)}")
vocab_file = Path(r"{str(vocab_file)}")
ref_audio = Path(r"{str(ref_audio)}")
out_wav = Path(r"{str(out_wav)}")
ref_text = {ref_text!r}
gen_text = {gen_text!r}

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device   =", device)
print("ckpt     =", ckpt_file)
print("vocab    =", vocab_file)
print("ref_audio=", ref_audio)
print("ref_text =", ref_text)
print("gen_text =", gen_text)

tts = F5TTS(
    model="F5TTS_EarlyBiMamba_v1",
    ckpt_file=str(ckpt_file),
    vocab_file=str(vocab_file),
    use_ema=True,
    device=device,
)

if hasattr(tts, "ema_model") and tts.ema_model is not None:
    tts.ema_model.float()
if hasattr(tts, "model") and tts.model is not None:
    tts.model.float()

tts.infer(
    ref_file=str(ref_audio),
    ref_text=ref_text,
    gen_text=gen_text,
    file_wave=str(out_wav),
    nfe_step=32,
    speed=1.0,
    seed=1234,
)

print("DONE:", out_wav)
"""

infer_env = dict(runtime_env if "runtime_env" in globals() else os.environ)
infer_env["MPLBACKEND"] = "Agg"

if "run_py" in globals():
    run_py(["-c", infer_script], cwd=REPO_DIR, env=infer_env)
else:
    subprocess.run(
        ["uv", "run", "--no-sync", "--python", str(VENV_PY), "python", "-c", infer_script],
        cwd=str(REPO_DIR),
        env=infer_env,
        check=True,
        text=True,
    )

print("Saved:", out_wav)
display(Audio(str(out_wav)))


## Notes
- Preset ini untuk Modal 1x H100 (bf16, single-process, non-smoke hyperparameters).
- Kalau OOM, turunkan datasets.batch_size_per_gpu bertahap (contoh: distill 19200 -> 16000 -> 12800, full 38400 -> 32000 -> 25600).
- Checkpoint hanya dibuat saat akhir fase distill dan akhir fase full training (tanpa checkpoint periodik).
- Full training tidak dari nol: inisialisasi partial load dari parent checkpoint yang sama dengan teacher distill (Eempostor/F5-TTS-INDO-FINETUNE-V2).
- Lokasi final checkpoint lokal: /root/modal-workdir/ckpts/distill_final_datasetku/model_last.pt dan /root/modal-workdir/ckpts/full_final_datasetku/model_last.pt.
- Cell upload HF akan membuat repo model baru yang private (exist_ok=False) lalu upload kedua checkpoint final.
- Secret env minimal yang wajib: KAGGLE_USERNAME, KAGGLE_KEY, WANDB_API_KEY, HF_TOKEN.
- Output inference test disimpan ke `/root/modal-workdir/infer_trainzeh4_modal_a100_40gb_test.wav`.
- Proses kerja utama ada di /root/modal-workdir.
